# Subspace Dynamics Analysis: Xp_1 vs Xp_2

Analyses of PSID/DPAD latent subspace properties across DBS ON/OFF conditions.

**Sections:**
1. Configuration & data loading
2. Behavioral DBS effect (raw kinematics)
3. Latent state extraction & trial-level statistics
4. A matrix structure & eigenvalue analysis (PSID only)
5. Power spectral density of latent states
6. C / Cz matrix loadings (PSID only)
7. Classifier comparison (mean-based vs covariance-based)
8. Summary table across subjects/sessions

In [1]:
import sys, os, pickle
from pathlib import Path
import numpy as np
import polars as pl
from scipy import signal, stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# -- Thesis color palette --
COLOR_ON = "#1f77b4"
COLOR_OFF = "#ff7f0e"
COLOR_XP1 = "#2ca02c"
COLOR_XP2 = "#d62728"
PLOTLY_TEMPLATE = "plotly_white"

## 1. Configuration

Define model runs to analyse. Each entry specifies model paths, data location, and parameters.
Set `model_type` to `"psid"` or `"dpad"`.

In [2]:
RUNS = [
    {
        "label": "PDI4 S2 (200Hz)",
        "model_type": "psid",
        "participant": "PDI4",
        "session": "2",
        "n1": 6,
        "nx": 30,
        "fs": 200,
        "data_root": "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band",
        "config_yaml": "training/setups/psid/narrow_band_200Hz/both/psid_behavioral_PDI4_2_nx_30_n6_i50_dbs_both_200Hz_narrow_band.yaml",
        "model_both": "results/psid_behavioral_PDI4_2_nx_30_n6_i50_dbs_both_200Hz_narrow_band/model_20260408_162132.pkl",
        "model_on": "results/psid_behavioral_PDI4_2_nx_30_n6_i50_dbs_on_200Hz_narrow_band/model_20260408_163407.pkl",
        "model_off": "results/psid_behavioral_PDI4_2_nx_30_n6_i50_dbs_off_200Hz_narrow_band/model_20260408_164031.pkl",
    },
    {
        "label": "PDI4 S3 (200Hz)",
        "model_type": "psid",
        "participant": "PDI4",
        "session": "3",
        "n1": 6,
        "nx": 25,
        "fs": 200,
        "data_root": "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band",
        "config_yaml": "training/setups/psid/narrow_band_200Hz/both/psid_behavioral_PDI4_3_nx_25_n6_i50_dbs_both_200Hz_narrow_band.yaml",
        "model_both": "results/psid_behavioral_PDI4_3_nx_25_n6_i50_dbs_both_200Hz_narrow_band/model_20260408_185522.pkl",
        "model_on": "results/psid_behavioral_PDI4_3_nx_25_n6_i50_dbs_on_200Hz_narrow_band/model_20260408_190749.pkl",
        "model_off": "results/psid_behavioral_PDI4_3_nx_25_n6_i50_dbs_off_200Hz_narrow_band/model_20260408_191423.pkl",
    },
    {
        "label": "PDI1 S4 (200Hz)",
        "model_type": "psid",
        "participant": "PDI1",
        "session": "4",
        "n1": 2,
        "nx": 15,
        "fs": 200,
        "data_root": "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band",
        "config_yaml": "training/setups/psid/narrow_band_200Hz/both/psid_behavioral_PDI1_4_nx_15_n2_i50_dbs_both_200Hz_narrow_band.yaml",
        "model_both": "results/psid_behavioral_PDI1_4_nx_15_n2_i50_dbs_both_200Hz_narrow_band/model_20260408_194919.pkl",
        "model_on": "results/psid_behavioral_PDI1_4_nx_15_n2_i50_dbs_on_200Hz_narrow_band/model_20260408_200052.pkl",
        "model_off": "results/psid_behavioral_PDI1_4_nx_15_n2_i50_dbs_off_200Hz_narrow_band/model_20260408_200652.pkl",
    },
    # To add DPAD runs, set "model_type": "dpad" -- sections 6 and 7 will auto-skip.
]

## Helper functions

In [3]:
# ── Data loading ────────────────────────────────────────────────────────────

import yaml as _yaml


def get_neural_channels_from_config(config_yaml):
    """Read the exact neural channel list from the training YAML config."""
    with open(config_yaml) as f:
        cfg = _yaml.safe_load(f)
    return cfg["data"]["channels"]["neural_input"]


def load_trials(data_root, participant, session, neural_channels):
    """Load all trials, return (Y_on, Y_off, Y_all, labels, df)."""
    base = Path(data_root) / f"participant_id={participant}" / f"session={session}"
    all_dfs = []
    for bdir in sorted(base.glob("block=*")):
        for f in sorted(bdir.glob("*.parquet")):
            all_dfs.append(pl.read_parquet(f))
    df = pl.concat(all_dfs)

    Y_on, Y_off, Y_all, labels = [], [], [], []
    for i in range(len(df)):
        stim = df["stim"][i]
        arrays = []
        for ch in neural_channels:
            vals = df[ch][i].to_numpy(allow_copy=True).astype(np.float64)
            arrays.append(vals)
        trial = np.column_stack(arrays)  # (T, n_channels)
        Y_all.append(trial)
        labels.append(1 if stim == "on" else 0)
        (Y_on if stim == "on" else Y_off).append(trial)

    return Y_on, Y_off, Y_all, np.array(labels), df


def load_model(path):
    """Load a pickled model (PSID LSSM or DPAD)."""
    with open(path, "rb") as f:
        model = pickle.load(f)
    if hasattr(model, "restoreModels"):
        model.restoreModels()
        model.set_steps_ahead([1])
        model.set_multi_step_with_data_gen(False)
    return model


def predict_model(model, Y_trials, model_type="psid"):
    """Run model.predict() handling DPAD padding."""
    if model_type == "psid":
        return model.predict(Y_trials)

    all_Zp, all_Yp, all_Xp = [], [], []
    block_samples = model.block_samples
    for y in Y_trials:
        T = y.shape[0]
        remainder = T % block_samples
        if remainder != 0:
            pad = np.zeros((block_samples - remainder, y.shape[1]))
            y_padded = np.concatenate([y, pad], axis=0)
        else:
            y_padded = y
        Zp, Yp, Xp = model.predict(y_padded)
        all_Zp.append(Zp[:T] if Zp is not None else None)
        all_Yp.append(Yp[:T] if Yp is not None else None)
        all_Xp.append(Xp[:T] if Xp is not None else None)
    return all_Zp, all_Yp, all_Xp

In [4]:
# ── Analysis functions ──────────────────────────────────────────────────────

BEHAV_COLS = ["tracing_velocity_x", "tracing_acceleration_magnitude"]

FREQ_BANDS = [
    (0, 4, "sub-theta (0-4)"),
    (4, 8, "theta (4-8)"),
    (8, 13, "alpha (8-13)"),
    (13, 30, "beta (13-30)"),
    (30, 50, "low-gamma (30-50)"),
    (50, 100, "high-gamma (50+)"),
]


def cohens_d(a, b):
    ps = np.sqrt((np.std(a) ** 2 + np.std(b) ** 2) / 2)
    return (np.mean(a) - np.mean(b)) / ps if ps > 0 else 0.0


def behavioral_dbs_effect(df):
    """Cohen's d and t-test for raw behavioral columns, ON vs OFF."""
    rows = []
    for col in BEHAV_COLS:
        if col not in df.columns:
            continue
        on_means = np.array(
            [
                np.nanmean(np.abs(np.array(df[col][i].to_list(), dtype=float)))
                for i in range(len(df))
                if df["stim"][i] == "on"
            ]
        )
        off_means = np.array(
            [
                np.nanmean(np.abs(np.array(df[col][i].to_list(), dtype=float)))
                for i in range(len(df))
                if df["stim"][i] == "off"
            ]
        )
        d = cohens_d(on_means, off_means)
        _, p = stats.ttest_ind(on_means, off_means)
        rows.append(
            {
                "feature": col.replace("tracing_", ""),
                "on_mean": np.mean(on_means),
                "off_mean": np.mean(off_means),
                "d": d,
                "p": p,
            }
        )
    return rows


def latent_trial_stats(Xp_list, labels, n1, nx):
    """Per-dimension Cohen's d of trial means for Xp_1 and Xp_2."""
    means = np.array([x.mean(axis=0) for x in Xp_list])
    on_mask, off_mask = labels == 1, labels == 0
    results = {"xp1": [], "xp2": []}
    for i in range(n1):
        d = cohens_d(means[on_mask, i], means[off_mask, i])
        _, p = stats.ttest_ind(means[on_mask, i], means[off_mask, i])
        results["xp1"].append({"dim": i, "d": d, "p": p})
    for i in range(n1, nx):
        d = cohens_d(means[on_mask, i], means[off_mask, i])
        _, p = stats.ttest_ind(means[on_mask, i], means[off_mask, i])
        results["xp2"].append({"dim": i, "d": d, "p": p})
    return results


def avg_psd(trials, dim_slice, fs):
    """Welch PSD averaged across trials and dimensions."""
    all_psd = []
    for trial in trials:
        x = trial[:, dim_slice]
        for d in range(x.shape[1]):
            f, pxx = signal.welch(x[:, d], fs=fs, nperseg=min(512, x.shape[0]))
            all_psd.append(pxx)
    return f, np.mean(all_psd, axis=0)


def psd_band_power(f, psd):
    """Compute mean power per frequency band."""
    return {
        name: float(np.mean(psd[(f >= lo) & (f < hi)])) for lo, hi, name in FREQ_BANDS
    }


def eigenvalue_modes(A, fs):
    """Extract oscillatory modes from A matrix eigenvalues."""
    eigs = np.linalg.eig(A)[0]
    modes = []
    for e in eigs:
        mag = np.abs(e)
        freq = np.abs(np.angle(e)) * fs / (2 * np.pi)
        decay_ms = -1000.0 / (fs * np.log(mag)) if 0 < mag < 1 else float("inf")
        modes.append(
            {
                "freq": freq,
                "mag": mag,
                "decay_ms": decay_ms,
                "is_complex": abs(np.imag(e)) > 1e-10,
            }
        )
    return sorted(modes, key=lambda m: -m["mag"])


def a_matrix_analysis(model_both, model_on, model_off, n1, nx, fs):
    """Full A-matrix block structure and eigenvalue comparison (PSID only)."""
    A = np.array(model_both.A)
    A_on, A_off = np.array(model_on.A), np.array(model_off.A)

    block_norms = {
        "A11 (Xp1->Xp1)": np.linalg.norm(A[:n1, :n1]),
        "A12 (Xp2->Xp1)": np.linalg.norm(A[:n1, n1:]),
        "A21 (Xp1->Xp2)": np.linalg.norm(A[n1:, :n1]),
        "A22 (Xp2->Xp2)": np.linalg.norm(A[n1:, n1:]),
    }
    diff_norms = {
        "A11 diff": np.linalg.norm(A_on[:n1, :n1] - A_off[:n1, :n1]),
        "A22 diff": np.linalg.norm(A_on[n1:, n1:] - A_off[n1:, n1:]),
        "A12 diff": np.linalg.norm(A_on[:n1, n1:] - A_off[:n1, n1:]),
        "A21 diff": np.linalg.norm(A_on[n1:, :n1] - A_off[n1:, :n1]),
        "Full A diff": np.linalg.norm(A_on - A_off),
    }
    modes = {
        "both": eigenvalue_modes(A, fs),
        "on": eigenvalue_modes(A_on, fs),
        "off": eigenvalue_modes(A_off, fs),
    }
    # Subspace-specific eigenvalues (ON vs OFF)
    sub_modes = {}
    for label, Am in [("on", A_on), ("off", A_off)]:
        sub_modes[f"{label}_xp1"] = eigenvalue_modes(Am[:n1, :n1], fs)
        sub_modes[f"{label}_xp2"] = eigenvalue_modes(Am[n1:, n1:], fs)

    return {
        "block_norms": block_norms,
        "diff_norms": diff_norms,
        "modes": modes,
        "sub_modes": sub_modes,
    }


def c_matrix_analysis(model, n1, neural_channels):
    """C and Cz matrix loading analysis (PSID only)."""
    C = np.array(model.C)
    Cz = np.array(model.Cz)

    norms_xp1 = np.linalg.norm(C[:, :n1], axis=1)
    norms_xp2 = np.linalg.norm(C[:, n1:], axis=1)

    # Group by electrode
    electrodes = sorted(set(ch.split("_")[1] for ch in neural_channels))
    by_electrode = {}
    for e in electrodes:
        mask = [i for i, ch in enumerate(neural_channels) if ch.split("_")[1] == e]
        by_electrode[f"ECOG_{e}"] = {
            "xp1": float(np.sum(norms_xp1[mask])),
            "xp2": float(np.sum(norms_xp2[mask])),
        }

    # Group by frequency band
    band_keywords = {
        "theta": "theta",
        "alpha": "alpha",
        "beta": "beta",
        "gamma": "gamma",
    }
    by_band = {}
    for bname, kw in band_keywords.items():
        mask = [i for i, ch in enumerate(neural_channels) if kw in ch]
        if mask:
            by_band[bname] = {
                "xp1": float(np.mean(norms_xp1[mask])),
                "xp2": float(np.mean(norms_xp2[mask])),
            }

    cz_xp1_norm = float(np.linalg.norm(Cz[:, :n1]))
    cz_xp2_norm = float(np.linalg.norm(Cz[:, n1:]))

    return {
        "by_electrode": by_electrode,
        "by_band": by_band,
        "cz_xp1": cz_xp1_norm,
        "cz_xp2": cz_xp2_norm,
    }


def classifier_comparison(Xp_list, labels, n1, nx, df):
    """Compare mean-based vs covariance-based classification on Xp_1 / Xp_2."""
    Xp1 = [x[:, :n1] for x in Xp_list]
    Xp2 = [x[:, n1:nx] for x in Xp_list]

    # Feature extraction
    def trial_means(trials):
        return np.array([x.mean(axis=0) for x in trials])

    def trial_stds(trials):
        return np.array([x.std(axis=0) for x in trials])

    def trial_cov(trials):
        feats = []
        for x in trials:
            c = np.cov(x.T)
            feats.append(c[np.triu_indices(c.shape[0])])
        return np.array(feats)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = LogisticRegression(max_iter=1000, C=1.0)
    results = {}

    for feat_name, extractor in [
        ("mean", trial_means),
        ("std", trial_stds),
        ("cov", trial_cov),
    ]:
        f1, f2 = extractor(Xp1), extractor(Xp2)
        s1 = cross_val_score(clf, f1, labels, cv=cv, scoring="balanced_accuracy")
        s2 = cross_val_score(clf, f2, labels, cv=cv, scoring="balanced_accuracy")
        results[feat_name] = {"xp1": float(np.mean(s1)), "xp2": float(np.mean(s2))}

    # Combined mean+std
    f1 = np.hstack([trial_means(Xp1), trial_stds(Xp1)])
    f2 = np.hstack([trial_means(Xp2), trial_stds(Xp2)])
    s1 = cross_val_score(clf, f1, labels, cv=cv, scoring="balanced_accuracy")
    s2 = cross_val_score(clf, f2, labels, cv=cv, scoring="balanced_accuracy")
    results["mean+std"] = {"xp1": float(np.mean(s1)), "xp2": float(np.mean(s2))}

    # Raw behavioral baseline
    raw_feats = []
    for i in range(len(df)):
        feats = []
        for col in BEHAV_COLS:
            if col in df.columns:
                vals = np.array(df[col][i].to_list(), dtype=float)
                vals = vals[~np.isnan(vals)]
                feats.extend([np.mean(vals), np.std(vals)])
        raw_feats.append(feats)
    raw_feats = np.array(raw_feats)
    if raw_feats.shape[1] > 0:
        s_raw = cross_val_score(
            clf, raw_feats, labels, cv=cv, scoring="balanced_accuracy"
        )
        results["raw_behavioral"] = {"value": float(np.mean(s_raw))}

    return results

In [5]:
# ── Plotting functions ──────────────────────────────────────────────────────


def plot_psd_comparison(f, psd_xp1_on, psd_xp1_off, psd_xp2_on, psd_xp2_off, title, fs):
    """4-panel PSD: Xp_1 ON/OFF and Xp_2 ON/OFF."""
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Xp_1 (behavioral)", "Xp_2 (non-behavioral)"],
        shared_yaxes=False,
    )
    for col, (psd_on, psd_off, label) in enumerate(
        [
            (psd_xp1_on, psd_xp1_off, "Xp_1"),
            (psd_xp2_on, psd_xp2_off, "Xp_2"),
        ],
        1,
    ):
        fig.add_trace(
            go.Scatter(x=f, y=psd_on, name=f"{label} ON", line=dict(color=COLOR_ON)),
            row=1,
            col=col,
        )
        fig.add_trace(
            go.Scatter(x=f, y=psd_off, name=f"{label} OFF", line=dict(color=COLOR_OFF)),
            row=1,
            col=col,
        )
    fig.update_xaxes(title_text="Frequency (Hz)", range=[0, fs / 2])
    fig.update_yaxes(type="log", title_text="Power")
    fig.update_layout(title=title, template=PLOTLY_TEMPLATE, height=400, width=900)
    return fig


def plot_eigenvalue_comparison(a_info, title):
    """Scatter plot of eigenvalue modes ON vs OFF, colored by subspace."""
    fig = make_subplots(
        rows=1, cols=2, subplot_titles=["Xp_1 eigenvalues", "Xp_2 eigenvalues"]
    )

    for col, sub in enumerate(["xp1", "xp2"], 1):
        for cond, color, symbol in [
            ("on", COLOR_ON, "circle"),
            ("off", COLOR_OFF, "diamond"),
        ]:
            modes = a_info["sub_modes"][f"{cond}_{sub}"]
            freqs = [m["freq"] for m in modes if m["is_complex"]]
            mags = [m["mag"] for m in modes if m["is_complex"]]
            decays = [m["decay_ms"] for m in modes if m["is_complex"]]
            fig.add_trace(
                go.Scatter(
                    x=freqs,
                    y=mags,
                    mode="markers",
                    marker=dict(color=color, symbol=symbol, size=10),
                    name=f"{cond.upper()}",
                    text=[f"decay={d:.0f}ms" for d in decays],
                    hovertemplate="%{x:.1f}Hz<br>|λ|=%{y:.5f}<br>%{text}",
                    showlegend=(col == 1),
                ),
                row=1,
                col=col,
            )

    fig.update_xaxes(title_text="Frequency (Hz)")
    fig.update_yaxes(title_text="|λ| (magnitude)", range=[0.97, 1.001])
    fig.update_layout(title=title, template=PLOTLY_TEMPLATE, height=400, width=900)
    return fig


def plot_c_matrix(c_info, title):
    """Bar chart of C matrix loadings by band and electrode."""
    fig = make_subplots(
        rows=1, cols=2, subplot_titles=["By electrode", "By frequency band"]
    )

    # Electrode
    electrodes = list(c_info["by_electrode"].keys())
    xp1_vals = [c_info["by_electrode"][e]["xp1"] for e in electrodes]
    xp2_vals = [c_info["by_electrode"][e]["xp2"] for e in electrodes]
    fig.add_trace(
        go.Bar(x=electrodes, y=xp1_vals, name="Xp_1", marker_color=COLOR_XP1),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=electrodes, y=xp2_vals, name="Xp_2", marker_color=COLOR_XP2),
        row=1,
        col=1,
    )

    # Band
    bands = list(c_info["by_band"].keys())
    xp1_b = [c_info["by_band"][b]["xp1"] for b in bands]
    xp2_b = [c_info["by_band"][b]["xp2"] for b in bands]
    fig.add_trace(
        go.Bar(x=bands, y=xp1_b, name="Xp_1", marker_color=COLOR_XP1, showlegend=False),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(x=bands, y=xp2_b, name="Xp_2", marker_color=COLOR_XP2, showlegend=False),
        row=1,
        col=2,
    )

    fig.update_yaxes(title_text="||C|| loading norm")
    fig.update_layout(
        title=title, template=PLOTLY_TEMPLATE, height=400, width=900, barmode="group"
    )
    return fig


def plot_classifier_comparison(cls_results, title):
    """Grouped bar chart of classifier balanced accuracy."""
    feat_names = [k for k in cls_results if k != "raw_behavioral"]
    xp1_vals = [cls_results[k]["xp1"] for k in feat_names]
    xp2_vals = [cls_results[k]["xp2"] for k in feat_names]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=feat_names, y=xp1_vals, name="Xp_1", marker_color=COLOR_XP1))
    fig.add_trace(go.Bar(x=feat_names, y=xp2_vals, name="Xp_2", marker_color=COLOR_XP2))

    if "raw_behavioral" in cls_results:
        raw_val = cls_results["raw_behavioral"]["value"]
        fig.add_hline(
            y=raw_val,
            line_dash="dash",
            line_color="black",
            annotation_text=f"Raw behavioral: {raw_val:.3f}",
        )

    fig.add_hline(y=0.5, line_dash="dot", line_color="gray", annotation_text="Chance")
    fig.update_yaxes(title_text="Balanced accuracy", range=[0.3, 1.0])
    fig.update_layout(
        title=title, template=PLOTLY_TEMPLATE, height=400, width=700, barmode="group"
    )
    return fig

## 2. Load data and models

Load trials and models for all configured runs. The `data` dict stores everything needed for subsequent analyses.

In [6]:
data = {}

for run in RUNS:
    label = run["label"]
    mtype = run["model_type"]
    n1, nx, fs = run["n1"], run["nx"], run["fs"]
    print(f"Loading {label} ...")

    neural_channels = get_neural_channels_from_config(run["config_yaml"])
    Y_on, Y_off, Y_all, labels, df = load_trials(
        run["data_root"], run["participant"], run["session"], neural_channels
    )

    model_both = load_model(run["model_both"])
    model_on = load_model(run["model_on"]) if "model_on" in run else None
    model_off = load_model(run["model_off"]) if "model_off" in run else None

    _, _, Xp_all = predict_model(model_both, Y_all, mtype)
    Xp_on = [Xp_all[i] for i in range(len(labels)) if labels[i] == 1]
    Xp_off = [Xp_all[i] for i in range(len(labels)) if labels[i] == 0]

    data[label] = {
        "run": run,
        "n1": n1,
        "nx": nx,
        "fs": fs,
        "model_type": mtype,
        "neural_channels": neural_channels,
        "df": df,
        "labels": labels,
        "model_both": model_both,
        "model_on": model_on,
        "model_off": model_off,
        "Xp_all": Xp_all,
        "Xp_on": Xp_on,
        "Xp_off": Xp_off,
    }
    print(
        f"  {sum(labels==1)} ON, {sum(labels==0)} OFF trials, {len(neural_channels)} channels, Xp shape: {Xp_all[0].shape}"
    )

print("\nAll runs loaded.")

Loading PDI4 S2 (200Hz) ...


  59 ON, 60 OFF trials, 60 channels, Xp shape: (2600, 30)
Loading PDI4 S3 (200Hz) ...


  71 ON, 46 OFF trials, 60 channels, Xp shape: (2600, 25)
Loading PDI1 S4 (200Hz) ...


  55 ON, 51 OFF trials, 60 channels, Xp shape: (2600, 15)

All runs loaded.


## 3. Behavioral DBS effect (raw kinematics)

Before looking at latent states, check how much DBS changes the raw behavioral outputs
(velocity_x, acceleration_magnitude). This establishes the ground truth: if DBS changes
behavior with large effect sizes, we'd expect the behavioral subspace (Xp_1) to carry
DBS-discriminative information.

In [7]:
for label, d in data.items():
    behav = behavioral_dbs_effect(d["df"])
    d["behavioral"] = behav
    print(f"\n--- {label} ---")
    for r in behav:
        sig = (
            "***"
            if r["p"] < 0.001
            else "**" if r["p"] < 0.01 else "*" if r["p"] < 0.05 else ""
        )
        print(
            f"  {r['feature']:<30s}: ON={r['on_mean']:10.2f}  OFF={r['off_mean']:10.2f}  d={r['d']:+.3f}  p={r['p']:.4f} {sig}"
        )


--- PDI4 S2 (200Hz) ---
  velocity_x                    : ON=    324.20  OFF=    438.58  d=-1.036  p=0.0000 ***
  acceleration_magnitude        : ON=  12718.89  OFF=  23804.28  d=-1.637  p=0.0000 ***

--- PDI4 S3 (200Hz) ---
  velocity_x                    : ON=    311.78  OFF=    438.33  d=-1.103  p=0.0000 ***
  acceleration_magnitude        : ON=  10389.52  OFF=  21018.49  d=-1.537  p=0.0000 ***

--- PDI1 S4 (200Hz) ---
  velocity_x                    : ON=    259.78  OFF=    237.50  d=+0.781  p=0.0001 ***
  acceleration_magnitude        : ON=   8018.52  OFF=   7836.32  d=+0.117  p=0.5489 


## 4. Latent state trial-level statistics

Do the latent states (Xp_1 and Xp_2) differ between DBS ON and OFF at the trial level?
We compute Cohen's d on per-trial means for each latent dimension. If Xp_1 captured the
behavioral DBS effect, we'd expect significant d values here.

In [8]:
for label, d in data.items():
    ls = latent_trial_stats(d["Xp_all"], d["labels"], d["n1"], d["nx"])
    d["latent_stats"] = ls

    print(f"\n--- {label} ---")
    print(f"  Xp_1 (behavioral, dims 0-{d['n1']-1}):")
    for r in ls["xp1"]:
        sig = "*" if r["p"] < 0.05 else ""
        print(f"    dim {r['dim']:2d}: d={r['d']:+.3f}  p={r['p']:.4f} {sig}")

    # Top 5 Xp_2 dims by |d|
    top_xp2 = sorted(ls["xp2"], key=lambda r: -abs(r["d"]))[:5]
    print(f"  Xp_2 (non-behavioral, top 5 by |d|):")
    for r in top_xp2:
        sig = "*" if r["p"] < 0.05 else ""
        print(f"    dim {r['dim']:2d}: d={r['d']:+.3f}  p={r['p']:.4f} {sig}")


--- PDI4 S2 (200Hz) ---
  Xp_1 (behavioral, dims 0-5):
    dim  0: d=-0.101  p=0.5842 
    dim  1: d=+0.162  p=0.3821 
    dim  2: d=-0.070  p=0.7071 
    dim  3: d=+0.247  p=0.1838 
    dim  4: d=+0.044  p=0.8140 
    dim  5: d=-0.038  p=0.8357 
  Xp_2 (non-behavioral, top 5 by |d|):
    dim 21: d=+0.310  p=0.0965 
    dim 16: d=-0.304  p=0.1025 
    dim 23: d=+0.294  p=0.1142 
    dim 18: d=+0.246  p=0.1850 
    dim 28: d=+0.192  p=0.3026 

--- PDI4 S3 (200Hz) ---
  Xp_1 (behavioral, dims 0-5):
    dim  0: d=-0.164  p=0.3884 
    dim  1: d=-0.023  p=0.9072 
    dim  2: d=-0.047  p=0.7985 
    dim  3: d=-0.020  p=0.9127 
    dim  4: d=+0.155  p=0.4066 
    dim  5: d=-0.034  p=0.8561 
  Xp_2 (non-behavioral, top 5 by |d|):
    dim 18: d=+0.410  p=0.0331 *
    dim  6: d=+0.303  p=0.1107 
    dim 19: d=+0.278  p=0.1473 
    dim 23: d=+0.214  p=0.2628 
    dim  8: d=-0.200  p=0.2918 

--- PDI1 S4 (200Hz) ---
  Xp_1 (behavioral, dims 0-1):
    dim  0: d=-0.223  p=0.2577 
    dim  1: d=-0.

## 5. Power spectral density of latent states

What frequency content lives in Xp_1 vs Xp_2? This reveals whether the behavioral
subspace captures slow kinematics vs neural oscillations, and how DBS modulates the
spectral content in each subspace.

In [9]:
for label, d in data.items():
    n1, nx, fs = d["n1"], d["nx"], d["fs"]
    f_psd, psd_xp1_on = avg_psd(d["Xp_on"], slice(0, n1), fs)
    _, psd_xp1_off = avg_psd(d["Xp_off"], slice(0, n1), fs)
    _, psd_xp2_on = avg_psd(d["Xp_on"], slice(n1, nx), fs)
    _, psd_xp2_off = avg_psd(d["Xp_off"], slice(n1, nx), fs)
    d["psd"] = {
        "f": f_psd,
        "xp1_on": psd_xp1_on,
        "xp1_off": psd_xp1_off,
        "xp2_on": psd_xp2_on,
        "xp2_off": psd_xp2_off,
    }

    fig = plot_psd_comparison(
        f_psd, psd_xp1_on, psd_xp1_off, psd_xp2_on, psd_xp2_off, f"PSD: {label}", fs
    )
    fig.show()

    # Band power ratios
    bp_on = psd_band_power(f_psd, psd_xp2_on)
    bp_off = psd_band_power(f_psd, psd_xp2_off)
    print(f"  Xp_2 band power ON/OFF ratios:")
    for band in bp_on:
        ratio = bp_on[band] / bp_off[band] if bp_off[band] > 0 else 0
        print(f"    {band}: {ratio:.3f}")

  Xp_2 band power ON/OFF ratios:
    sub-theta (0-4): 0.973
    theta (4-8): 0.980
    alpha (8-13): 0.895
    beta (13-30): 0.861
    low-gamma (30-50): 0.931
    high-gamma (50+): 1.147


  Xp_2 band power ON/OFF ratios:
    sub-theta (0-4): 1.004
    theta (4-8): 1.114
    alpha (8-13): 0.901
    beta (13-30): 0.796
    low-gamma (30-50): 1.095
    high-gamma (50+): 1.282


  Xp_2 band power ON/OFF ratios:
    sub-theta (0-4): 0.815
    theta (4-8): 0.899
    alpha (8-13): 0.921
    beta (13-30): 0.968
    low-gamma (30-50): 4.545
    high-gamma (50+): 1.052


## 6. A matrix analysis (PSID only)

The state transition matrix A governs how latent states evolve: $x_{t+1} = A x_t$.

**Block structure**: A11 (Xp_1 self-dynamics), A12 (Xp_2 -> Xp_1 coupling), A21 (Xp_1 -> Xp_2),
A22 (Xp_2 self-dynamics). If A12 ~ 0, the behavioral subspace is decoupled from neural dynamics.

**Eigenvalues**: Complex eigenvalues reveal oscillatory modes (frequency = angle, |lambda| = persistence).
Comparing ON vs OFF models shows which modes DBS modulates.

**Subspace eigenvalues**: Eigenvalues of A11 and A22 blocks separately show the intrinsic dynamics
of each subspace.

In [10]:
for label, d in data.items():
    if d["model_type"] != "psid" or d["model_on"] is None:
        print(f"  {label}: skipped (not PSID or no ON/OFF models)")
        continue

    a_info = a_matrix_analysis(
        d["model_both"], d["model_on"], d["model_off"], d["n1"], d["nx"], d["fs"]
    )
    d["a_matrix"] = a_info

    print(f"\n--- {label} ---")
    print(f"  Block structure (Frobenius norms):")
    for k, v in a_info["block_norms"].items():
        print(f"    {k}: {v:.4f}")
    coupling = a_info["block_norms"]["A12 (Xp2->Xp1)"]
    print(
        f"    --> A12 = {coupling:.6f}  ({'DECOUPLED' if coupling < 0.01 else 'coupled'})"
    )

    print(f"\n  ON vs OFF difference (Frobenius norms):")
    for k, v in a_info["diff_norms"].items():
        print(f"    {k}: {v:.4f}")

    fig = plot_eigenvalue_comparison(a_info, f"Eigenvalues ON vs OFF: {label}")
    fig.show()


--- PDI4 S2 (200Hz) ---
  Block structure (Frobenius norms):
    A11 (Xp1->Xp1): 2.4468
    A12 (Xp2->Xp1): 0.0000
    A21 (Xp1->Xp2): 0.0221
    A22 (Xp2->Xp2): 4.8615
    --> A12 = 0.000000  (DECOUPLED)

  ON vs OFF difference (Frobenius norms):
    A11 diff: 0.1910
    A22 diff: 4.9486
    A12 diff: 0.0000
    A21 diff: 0.0335
    Full A diff: 4.9524



--- PDI4 S3 (200Hz) ---
  Block structure (Frobenius norms):
    A11 (Xp1->Xp1): 2.4476
    A12 (Xp2->Xp1): 0.0000
    A21 (Xp1->Xp2): 0.0789
    A22 (Xp2->Xp2): 4.2944
    --> A12 = 0.000000  (DECOUPLED)

  ON vs OFF difference (Frobenius norms):
    A11 diff: 0.3142
    A22 diff: 6.8349
    A12 diff: 0.0000
    A21 diff: 0.0839
    Full A diff: 6.8427



--- PDI1 S4 (200Hz) ---
  Block structure (Frobenius norms):
    A11 (Xp1->Xp1): 1.4128
    A12 (Xp2->Xp1): 0.0000
    A21 (Xp1->Xp2): 0.0040
    A22 (Xp2->Xp2): 3.4773
    --> A12 = 0.000000  (DECOUPLED)

  ON vs OFF difference (Frobenius norms):
    A11 diff: 0.1642
    A22 diff: 4.2469
    A12 diff: 0.0000
    A21 diff: 0.0062
    Full A diff: 4.2500


## 7. C and Cz matrix loadings (PSID only)

**C matrix** (observation): maps latent states to neural observations ($y_t = C x_t$).
The loading norms show which neural channels (electrodes, frequency bands) are captured
by Xp_1 vs Xp_2.

**Cz matrix** (behavioral output): maps latent states to behavioral predictions ($z_t = C_z x_t$).
If Cz[:, n1:] ~ 0, behavioral prediction comes entirely from Xp_1.

In [11]:
for label, d in data.items():
    if d["model_type"] != "psid":
        print(f"  {label}: skipped (not PSID)")
        continue

    c_info = c_matrix_analysis(d["model_both"], d["n1"], d["neural_channels"])
    d["c_matrix"] = c_info

    print(f"\n--- {label} ---")
    print(
        f"  Cz norms: Xp_1={c_info['cz_xp1']:.4f}, Xp_2={c_info['cz_xp2']:.4f} "
        f"(ratio={c_info['cz_xp1']/c_info['cz_xp2']:.1f}x)"
    )
    print(f"  C loading by frequency band:")
    for b, v in c_info["by_band"].items():
        ratio = v["xp1"] / v["xp2"] if v["xp2"] > 0 else float("inf")
        print(
            f"    {b:<10s}: Xp_1={v['xp1']:.4f}, Xp_2={v['xp2']:.4f}, ratio={ratio:.3f}"
        )

    fig = plot_c_matrix(c_info, f"C matrix loadings: {label}")
    fig.show()


--- PDI4 S2 (200Hz) ---
  Cz norms: Xp_1=0.1715, Xp_2=0.0050 (ratio=34.4x)
  C loading by frequency band:
    theta     : Xp_1=0.0446, Xp_2=0.5802, ratio=0.077
    alpha     : Xp_1=0.0289, Xp_2=0.7277, ratio=0.040
    beta      : Xp_1=0.0064, Xp_2=0.5724, ratio=0.011
    gamma     : Xp_1=0.0029, Xp_2=0.3864, ratio=0.007



--- PDI4 S3 (200Hz) ---
  Cz norms: Xp_1=0.1452, Xp_2=0.0132 (ratio=11.0x)
  C loading by frequency band:
    theta     : Xp_1=0.1904, Xp_2=0.4406, ratio=0.432
    alpha     : Xp_1=0.0989, Xp_2=0.5257, ratio=0.188
    beta      : Xp_1=0.0146, Xp_2=0.4337, ratio=0.034
    gamma     : Xp_1=0.0057, Xp_2=0.3260, ratio=0.018



--- PDI1 S4 (200Hz) ---
  Cz norms: Xp_1=0.2094, Xp_2=0.0184 (ratio=11.4x)
  C loading by frequency band:
    theta     : Xp_1=0.0322, Xp_2=0.2658, ratio=0.121
    alpha     : Xp_1=0.0028, Xp_2=0.3127, ratio=0.009
    beta      : Xp_1=0.0013, Xp_2=0.2902, ratio=0.004
    gamma     : Xp_1=0.0007, Xp_2=0.2139, ratio=0.003


## 7b. Channel importance: Behavioral vs Neural relevance

**Cy row norms** (eigenvalue-weighted) measure how much each neural channel couples to the latent dynamics — analogous to communality in factor analysis.

**Cy→Cz combined** (`||Cz @ Cy[i,:]||`) traces the full path: neural channel → latent states → behavioral output. This identifies which channels ultimately drive behavior through the learned latent space.

Top-5 channels are automatically extracted for each criterion.

In [ ]:
from scripts.extract_psid_channel_importance import (
    get_top_behavioral_and_neural,
    compute_behavioral_relevance,
    extract_channel_importance,
)

for label, d in data.items():
    if d["model_type"] != "psid":
        print(f"  {label}: skipped (not PSID)")
        continue

    model_path = d["run"]["model_both"]
    n1, nx = d["n1"], d["nx"]
    channels = d["neural_channels"]

    result = get_top_behavioral_and_neural(model_path, n1, channels, top_n=5)
    d["channel_importance"] = result

    print(f"\n{'='*60}")
    print(f"  {label}  (nx={nx}, n1={n1})")
    print(f"{'='*60}")

    print(f"\n  Top 5 BEHAVIORAL channels (||Cz @ Cy[i,:]||):")
    for i, (ch, sc) in enumerate(
        zip(result["top_behavioral"], result["top_behavioral_scores"])
    ):
        parts = ch.split("_", 2)
        print(f"    {i+1}. E{parts[1]} {parts[2]:30s}  {sc:.6f}")

    print(f"\n  Top 5 NEURAL channels (eigenvalue-weighted ||Cy||):")
    for i, (ch, sc) in enumerate(
        zip(result["top_neural"], result["top_neural_scores"])
    ):
        parts = ch.split("_", 2)
        print(f"    {i+1}. E{parts[1]} {parts[2]:30s}  {sc:.6f}")

    # Show Cz structure: how much each behavioral output loads on x1 vs x2
    imp = result["importance"]
    Cz = imp["Cz"]
    if Cz is not None:
        print(f"\n  Cz structure (latent -> behavior):")
        for z_idx, z_name in enumerate(["velocity_x", "accel_mag"]):
            row = Cz[z_idx, :]
            x1_norm = np.linalg.norm(row[:n1])
            x2_norm = np.linalg.norm(row[n1:])
            ratio = x1_norm / x2_norm if x2_norm > 0 else float("inf")
            print(
                f"    {z_name}: x1={x1_norm:.4f}, x2={x2_norm:.4f} (ratio={ratio:.1f}x)"
            )

In [ ]:
# Visualize behavioral vs neural channel importance
for label, d in data.items():
    if d["model_type"] != "psid" or "channel_importance" not in d:
        continue

    result = d["channel_importance"]
    imp = result["importance"]
    br = result["behavioral_relevance"]
    channels = d["neural_channels"]
    n1 = d["n1"]

    # Short labels: E1_theta_4_8 etc.
    short_labels = [
        ch.replace("ECOG_", "E").replace("_raw", "").replace("_env", "")
        for ch in channels
    ]

    # Scatter: behavioral relevance vs neural importance
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            f"Behavioral vs Neural importance",
            f"Top channels by criterion",
        ],
        horizontal_spacing=0.12,
    )

    neural_scores = imp["weighted_importance"]
    behav_scores = br["behavioral_relevance"]

    # Color by electrode
    colors = []
    electrode_colors = {"1": "#1f77b4", "2": "#ff7f0e", "3": "#2ca02c", "4": "#d62728"}
    for ch in channels:
        e = ch.split("_")[1]
        colors.append(electrode_colors.get(e, "gray"))

    fig.add_trace(
        go.Scatter(
            x=neural_scores,
            y=behav_scores,
            mode="markers",
            text=short_labels,
            marker=dict(color=colors, size=8, opacity=0.7),
            hovertemplate="%{text}<br>Neural: %{x:.4f}<br>Behavioral: %{y:.6f}",
            showlegend=False,
        ),
        row=1,
        col=1,
    )
    fig.update_xaxes(title_text="Neural importance (eig-weighted ||Cy||)", row=1, col=1)
    fig.update_yaxes(title_text="Behavioral relevance (||Cz @ Cy||)", row=1, col=1)

    # Bar chart: top 5 behavioral + top 5 neural
    top_beh = result["top_behavioral"]
    top_neur = result["top_neural"]
    top_beh_short = [
        ch.replace("ECOG_", "E").replace("_raw", "").replace("_env", "")
        for ch in top_beh
    ]
    top_neur_short = [
        ch.replace("ECOG_", "E").replace("_raw", "").replace("_env", "")
        for ch in top_neur
    ]

    # Normalize scores to [0, 1] for comparison
    beh_norm = np.array(result["top_behavioral_scores"])
    beh_norm = beh_norm / beh_norm.max() if beh_norm.max() > 0 else beh_norm
    neur_norm = np.array(result["top_neural_scores"])
    neur_norm = neur_norm / neur_norm.max() if neur_norm.max() > 0 else neur_norm

    all_labels = top_beh_short + [""] + top_neur_short
    all_scores = list(beh_norm) + [0] + list(neur_norm)
    all_colors = ["#2196F3"] * 5 + ["white"] + ["#FF9800"] * 5

    fig.add_trace(
        go.Bar(
            y=all_labels,
            x=all_scores,
            orientation="h",
            marker_color=all_colors,
            text=["Behavioral"] * 5 + [""] + ["Neural"] * 5,
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.update_xaxes(title_text="Normalized importance", row=1, col=2)

    fig.update_layout(
        title=f"Channel importance: {label}",
        template=PLOTLY_TEMPLATE,
        height=450,
        width=1000,
    )
    fig.show()

## 8. Classifier comparison

The CSP+LDA pipeline used for classification is **covariance-based** and ignores trial means.
Here we compare different feature types to understand which aspect of the latent states
carries DBS information:

- **mean**: per-trial average of each latent dimension (detects mean shifts)
- **std**: per-trial standard deviation (detects variance changes)
- **cov**: upper triangle of per-trial covariance matrix (what CSP uses)
- **mean+std**: combined
- **raw behavioral**: baseline using raw velocity/acceleration (not through the model)

If Xp_1 means are at chance but raw behavioral classifies well, the model's state-space
mapping discards the between-condition mean shift.

In [12]:
for label, d in data.items():
    cls = classifier_comparison(d["Xp_all"], d["labels"], d["n1"], d["nx"], d["df"])
    d["classifiers"] = cls

    print(f"\n--- {label} ---")
    print(f"  {'Features':<18} {'Xp_1':>8} {'Xp_2':>8}")
    print(f"  {'-'*36}")
    for k, v in cls.items():
        if k == "raw_behavioral":
            print(f"  {'raw behavioral':<18} {v['value']:>8.4f}    (baseline)")
        else:
            print(f"  {k:<18} {v['xp1']:>8.4f} {v['xp2']:>8.4f}")

    fig = plot_classifier_comparison(cls, f"Classifier comparison: {label}")
    fig.show()


--- PDI4 S2 (200Hz) ---
  Features               Xp_1     Xp_2
  ------------------------------------
  mean                 0.4947   0.4780
  std                  0.6636   0.7061
  cov                  0.6227   0.7394
  mean+std             0.6636   0.7061
  raw behavioral       0.8053    (baseline)



--- PDI4 S3 (200Hz) ---
  Features               Xp_1     Xp_2
  ------------------------------------
  mean                 0.5000   0.5000
  std                  0.4933   0.8640
  cov                  0.5197   0.9329
  mean+std             0.4933   0.8640
  raw behavioral       0.8986    (baseline)



--- PDI1 S4 (200Hz) ---
  Features               Xp_1     Xp_2
  ------------------------------------
  mean                 0.5000   0.5000
  std                  0.6236   0.6327
  cov                  0.6555   0.6418
  mean+std             0.6236   0.6327
  raw behavioral       0.7064    (baseline)


## 9. Cross-run summary

Aggregate key metrics across all runs for comparison.

In [13]:
print(
    f"{'Run':<22} {'Behav d':>8} {'A12':>8} {'Cz ratio':>9} {'Cls mean':>10} {'Cls cov':>10} {'Raw beh':>8}"
)
print(
    f"{'':22s} {'(vel_x)':>8} {'norm':>8} {'Xp1/Xp2':>9} {'Xp1/Xp2':>10} {'Xp1/Xp2':>10}"
)
print("-" * 85)

for label, d in data.items():
    # Behavioral d for velocity_x
    vel_d = next(
        (r["d"] for r in d.get("behavioral", []) if "velocity_x" in r["feature"]),
        float("nan"),
    )

    # A12 norm
    a12 = (
        d.get("a_matrix", {}).get("block_norms", {}).get("A12 (Xp2->Xp1)", float("nan"))
    )

    # Cz ratio
    c_info = d.get("c_matrix", {})
    cz_ratio = (
        c_info.get("cz_xp1", 0) / c_info.get("cz_xp2", 1) if c_info else float("nan")
    )

    # Classifier results
    cls = d.get("classifiers", {})
    mean_str = (
        f"{cls['mean']['xp1']:.2f}/{cls['mean']['xp2']:.2f}" if "mean" in cls else "?"
    )
    cov_str = (
        f"{cls['cov']['xp1']:.2f}/{cls['cov']['xp2']:.2f}" if "cov" in cls else "?"
    )
    raw_str = (
        f"{cls['raw_behavioral']['value']:.2f}" if "raw_behavioral" in cls else "?"
    )

    print(
        f"{label:<22} {vel_d:+8.3f} {a12:8.4f} {cz_ratio:9.1f}x {mean_str:>10} {cov_str:>10} {raw_str:>8}"
    )

Run                     Behav d      A12  Cz ratio   Cls mean    Cls cov  Raw beh
                        (vel_x)     norm   Xp1/Xp2    Xp1/Xp2    Xp1/Xp2
-------------------------------------------------------------------------------------
PDI4 S2 (200Hz)          -1.036   0.0000      34.4x  0.49/0.48  0.62/0.74     0.81
PDI4 S3 (200Hz)          -1.103   0.0000      11.0x  0.50/0.50  0.52/0.93     0.90
PDI1 S4 (200Hz)          +0.781   0.0000      11.4x  0.50/0.50  0.66/0.64     0.71


## Interpretation guide

### Section 3: Behavioral DBS effect
- **Cohen's d** measures the standardized mean difference between DBS ON and OFF.
- |d| > 0.8 is a large effect. If behavior changes strongly with DBS (e.g., PDI4 velocity d > 1.0), we'd naively expect the behavioral subspace Xp_1 to carry that information.
- **Key question**: does the behavioral DBS effect survive the model's latent representation?

### Section 4: Latent trial-level statistics
- Per-dimension Cohen's d on **trial means** of latent states.
- If all Xp_1 dimensions have d ~ 0 despite large behavioral effects, the model does not preserve the between-condition mean shift in the behavioral subspace.
- This can happen because the state-space model captures **temporal dynamics** (autocorrelation), not static mean levels. The mean shift gets lost in the A matrix evolution.

### Section 5: Power spectral density
- **Xp_1 frequency content**: if Xp_1 only has power below ~4 Hz, it captures slow kinematics (movement trajectory), not neural oscillations. This means Xp_1 is driven by the behavioral outputs, not by shared neural-behavioral dynamics.
- **Xp_2 ON/OFF PSD ratios**: beta (13-30 Hz) suppression with DBS ON (ratio < 1) is the classic electrophysiological signature of DBS in Parkinson's disease. High-gamma enhancement (ratio > 1) is also commonly reported.
- If Xp_1 and Xp_2 have non-overlapping spectral content, the subspaces are dynamically decoupled.

### Section 6: A matrix analysis (PSID only)
- **Block structure**: A12 (Xp_2 -> Xp_1 coupling) near zero means the non-behavioral subspace has no causal influence on the behavioral subspace. Information cannot flow from Xp_2 to Xp_1.
- **ON vs OFF difference**: if A22 diff >> A11 diff, DBS modulates neural dynamics (Xp_2) far more than behavioral dynamics (Xp_1).
- **Eigenvalue plot**: each dot is an oscillatory mode. Higher |lambda| = more persistent. Compare ON vs OFF mode frequencies — shifted peaks indicate DBS modulation of specific oscillatory bands. Shorter decay times (lower |lambda|) with DBS ON suggest increased neural damping.

### Section 7: C and Cz matrix loadings (PSID only)
- **Cz ratio (Xp_1/Xp_2)**: if >> 1, behavioral prediction comes almost entirely from Xp_1 — PSID cleanly separated behavioral and neural subspaces.
- **C loadings by band**: shows which neural frequency bands are captured by each subspace. If Xp_1 has near-zero C loading across all bands, neural signals essentially bypass Xp_1 entirely.
- **Implication**: PSID optimizes Xp_1 to reconstruct behavior from kinematics alone, not from neural activity. This is the fundamental limitation — Xp_1 is not 'behaviorally-relevant neural dynamics' but 'behavioral output reconstruction'.

### Section 8: Classifier comparison
- **mean features at chance** for Xp_1 confirms the between-condition mean shift is lost in the latent space.
- **std/cov features** capture variance and temporal structure differences — these may still carry some DBS information even in Xp_1 (but typically less than Xp_2).
- **raw behavioral baseline** shows how well DBS can be classified from raw kinematics alone. If this is high (e.g., 0.82) but Xp_1 mean is at chance, the model discards usable information.
- **CSP+LDA** (your main pipeline classifier) is covariance-based, equivalent to the 'cov' row here.

### Section 9: Cross-run summary
- Compare across sessions to check if findings are consistent.
- Key columns: **Behav d** (how much DBS changes behavior), **A12** (subspace coupling), **Cz ratio** (behavioral output mapping), **Cls cov** (what CSP+LDA would give), **Raw beh** (ceiling).

### Overall narrative for PSID
PSID finds the mathematically optimal linear decomposition for behavioral prediction, but this does not correspond to 'neural dynamics that drive behavior'. Instead:
- **Xp_1** = pure kinematic reconstruction (~1-4 Hz), decoupled from neural signals
- **Xp_2** = all neural dynamics (theta through gamma), carries DBS-sensitive oscillatory modulation
- The subspaces are structurally decoupled (A12=0, C loadings ~0 for Xp_1)

This is likely because neural band power (4-80 Hz) and behavioral kinematics (~1-4 Hz) operate at different timescales that a linear state-space model cannot bridge. DPAD (nonlinear) may find cross-frequency coupling that PSID misses — compare by adding DPAD entries to RUNS.
